### Plot velocity profiles and compare to experimental data
The experimental data can be found in `validation_exp_data` and is taken from
*F. Grossi, M. Braza, and Y. Hoarau, “Prediction of transonic buffet by delayed detached-eddy simulation,”
https://doi.org/10.2514/1.J052873.*

In [41]:
import matplotlib.pyplot as plt

from os import makedirs
from glob import glob
from pandas import read_csv
from os.path import join, exists

In [37]:
# validation @ Ma = 0.73
u_inf = 242.16629

# chord length
chord = 1

# use latex fonts
plt.style.use("default")
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360})

# use these line styles
ls = ["-", "--", "-.", ":"]

In [111]:
# SALSA vs. exp. data
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.73_Re3e6")
save_dir = join("..", "run", "plots", "URANS_validation", "URANS_blockMesh", "SALSA", "revised_new_mesh", "comparison_velocity_profiles")
cases = ["URANS_SA_alpha3.5deg_blockMesh_newMesh", "URANS_SALSA_alpha3.5deg_blockMesh_useRmod_useSmod_newMesh"]

legend = [r"$\mathrm{SA}$", r"$\mathrm{SALSA}$"]

In [92]:
# path to the experimental data from Grossi et al. / Deck
locations = ["0.28", "0.45", "0.6", "0.75"]
exp_path_mean = [join("..", "validation_exp_data", f"Grossi_Ux_mean_xc{loc}.csv") for loc in locations]
exp_path_rms = [join("..", "validation_exp_data", f"Grossi_Ux_rms_xc{loc}.csv") for loc in locations]

# load data
mean_data_exp = [read_csv(p, sep=",", comment="#", header=None, usecols=[0, 1], names=["u_Uinf", "yc"]) for p in exp_path_mean]
rms_data_exp = [read_csv(p, sep=",", comment="#", header=None, usecols=[0, 1], names=["u_Uinf", "yc"]) for p in exp_path_rms]

In [112]:
# load the numerical data, only load the last write time since we are interested in mean / prime2Mean
names = ["z", "UMean_x", "UPrime2Mean_x"]
loc = ["028", "045", "06", "075"]
lines = []
for c in cases:
    files = [glob(join(load_dir, c, "postProcessing", "sample_lines", "*", f"xc_{l}_*.csv"))[-1] for l in loc]
    line = [read_csv(f, names=names, header=None, sep=",", skiprows=1, usecols=[0, 9, 12]) for f in files]
    lines.append(line)

In [ ]:
# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [ ]:
# plot mean velocity profiles and compare to experimental data
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")

# add plot for experimental data
for i, d in enumerate(mean_data_exp):
    if i == 0:
        ax[i].plot(d["u_Uinf"], d["yc"], color="black", marker=".", fillstyle="none", linestyle="none", label=r"$\mathrm{exp.}$")
    else:
        ax[i].plot(d["u_Uinf"], d["yc"], color="black", marker=".", linestyle="none", fillstyle="none")

    ax[i].set_xlim(-0.02, 1.8)
    ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
    ax[i].minorticks_on()
    ax[i].tick_params(axis="both", which="minor", bottom=True)
    ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
    ax[i].set_title(fr"$ x / c = {locations[i]}$")

for j in range(len(cases)):
    for i in range(len(mean_data_exp)):
        if i == 0:
            # since the airfoil surface is not at z = 0, we have to shift it
            ax[i].plot(lines[j][i]["UMean_x"].values / u_inf, (lines[j][i]["z"].values - lines[j][i]["z"][0]) / chord, zorder=10, color="black",
                       marker="none", ls=ls[j], label=legend[j])
        else:
            ax[i].plot(lines[j][i]["UMean_x"].values / u_inf, (lines[j][i]["z"].values - lines[j][i]["z"][0]) / chord, zorder=10, color="black",
                       marker="none", ls=ls[j])

fig.supxlabel(r"$\bar{u} / U_\infty$")
ax[0].set_ylabel(r"$y / c$")
ax[0].set_ylim(0, 0.06)
fig.tight_layout()
fig.legend(ncol=4, loc="upper center")
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, f"comparison_mean_velocities.png"))
plt.show()

In [ ]:
# plot rms velocity profiles and compare to experimental data
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")

# add plot for experimental data
for i, d in enumerate(rms_data_exp):
    if i == 0:
        ax[i].plot(d["u_Uinf"], d["yc"], zorder=10, color="black", marker=".", fillstyle="none", linestyle="none", label=r"$\mathrm{exp.}$")
    else:
        ax[i].plot(d["u_Uinf"], d["yc"], zorder=10, color="black", marker=".", linestyle="none", fillstyle="none")
    ax[i].set_xlim(0.0, 0.8)
    ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
    ax[i].minorticks_on()
    ax[i].tick_params(axis="both", which="minor", bottom=True)
    ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
    ax[i].set_title(fr"$ x / c = {locations[i]}$")

for j in range(len(cases)):
    for i in range(len(mean_data_exp)):
        if i == 0:
            # since the airfoil surface is not at z = 0, we have to shift it
            ax[i].plot(lines[j][i]["UPrime2Mean_x"].pow(0.5).values / u_inf, (lines[j][i]["z"].values - lines[j][i]["z"][0]) / chord, zorder=10, color="black",
                       marker="none", ls=ls[j], label=legend[j])
        else:
            ax[i].plot(lines[j][i]["UPrime2Mean_x"].pow(0.5).values / u_inf, (lines[j][i]["z"].values - lines[j][i]["z"][0]) / chord, zorder=10, color="black",
                       marker="none", ls=ls[j])

fig.supxlabel(r"$u_\mathrm{RMS} / U_\infty$")
ax[0].set_ylabel(r"$y / c$")
ax[0].set_ylim(0, 0.06)
fig.tight_layout()
fig.legend(ncol=4, loc="upper center")
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, f"comparison_rms_velocities.png"))
plt.show()